# **Programming Assignment 1**

In [30]:
import os
from zipfile import ZipFile
if not os.path.exists('cache'):
  os.mkdir('cache')


In [31]:
with ZipFile("C:\\Users\mange\\Downloads\\fma_metadata.zip", 'r') as f:
  f.extractall('cache')

# Locality Sensitive Hashing (LSH)

The goal of this task was to implement an approximate nearest neighbor (ANN) classifier for music genre prediction using Locality Sensitive Hashing (LSH) based on random projections. The classifier identifies similar tracks in a high‑dimensional feature space and predicts the genre of new tracks based on their nearest neighbors.

*Subtask 1: General functionality*

In [7]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
# # Load track metadata
df_tracks = pd.read_csv("C:\\Users\\mange\\cache\\fma_metadata\\tracks.csv",index_col=0, header = [0,1])
# # Load feature metadata
df_features = pd.read_csv("C:\\Users\\mange\\cache\\fma_metadata\\features.csv",index_col=0, header = [0,1])
#df_genres = pd.read_csv("fma_metadata/fma_metadata/genres.csv", index_col=0, header = [0,1])

C:\Users\mange\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3058: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [13]:
# Keep only medium subset
df_tracks = df_tracks[df_tracks[("set", "subset")] == "medium"]

df_tracks.shape

(17000, 52)

In [14]:
# Align features with selected track IDs
match_id = df_tracks.index.intersection(df_features.index)

df_tracks = df_tracks.loc[match_id]
df_features = df_features.loc[match_id]

In [15]:

# Extract predefined split
splits = df_tracks[('set'),('split')]
genres = df_tracks[('track'), ('genre_top')]


In [16]:
# Required 8 genres
top_genres = ["Hip-Hop","Pop", "Folk", "Rock", "Experimental", "International", "Electronic", "Instrumental"]

# Extract the genre column from df_tracks
genres = df_tracks[('track', 'genre_top')]

# Filter only the 8 target genres
mask = genres.isin(top_genres)
df_tracks = df_tracks[mask]

print("Tracks:", df_tracks.shape)
print("features:", df_features.shape)

Tracks: (14805, 52)
features: (16791, 518)


In [17]:
# split thed data
training = df_tracks.index[df_tracks['set', 'split'] == 'training']
testing = df_tracks.index[df_tracks['set', 'split'] == 'test']
validation = df_tracks.index[df_tracks['set', 'split'] == 'validation']

In [18]:
# Train-Test-Validate data
X_train = df_features.loc[training].values
X_test = df_features.loc[testing].values
X_val = df_features.loc[validation].values

y_train  = genres.loc[training]
y_test = genres.loc[testing]
y_val = genres.loc[validation]

In [19]:
# Standardization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

Our dataset was constructed from the FMA metadata files tracks.csv and features.csv.

We filtered the medium subset, aligned the track identifiers across both files, and restricted the data to the eight required genres.

The predefined FMA split was used to obtain training, validation, and test sets.

 All feature vectors were then standardized using StandardScaler to ensure uniform scaling

In [20]:
# Achlioptas Matrix
def achlioptas_matrix(d, l):
    #Construct random projection matrix R using Achlioptas scheme:
        #    rij = +sqrt(3) with prob 1/6
        #           0        with prob 2/3
        #          -sqrt(3) with prob 1/6
    probs = np.random.rand(d, l)
    R = np.zeros((d, l))
    R[probs < 1/6] = np.sqrt(3)
    R[(probs >= 1/6) & (probs < 5/6)] = 0
    R[probs >= 5/6] = -np.sqrt(3)
    return R

In [21]:
# LSH Class
class LSH:
    def __init__(self, l,n):
        self.l = l
        self.n = n
        self.hash_tables = []
        self.projections = []

    def fit(self, X):
        # Builds n_tables hash tables
        d = X.shape[1]
        for _ in range(self.n):
            R = achlioptas_matrix(self.l, d)
            table = {}
            for idx, x in enumerate(X):
                h = tuple((R @ x > 0).astype(int))
                table.setdefault(h, []).append(idx)

            self.hash_tables.append(table)
            self.projections.append(R)


    def query(self, x):
        #  Return list of candidate neighbours
        candidates = set()
        for i in range(self.n):
            R = self.projections[i]
            table = self.hash_tables[i]

            h = tuple((R@x>0).astype(int))
            if h in table:
                candidates.update(table[h])
        return list(candidates)

In [24]:
# similarity function
def sim_metric(x, X_candidates, metric):
    if metric == "cosine":
        return cosine_similarity(x.reshape(1, -1), X_candidates)[0] 
    elif metric == "euclidean":
        distan = np.linalg.norm(X_candidates - x, axis=1)
        return -distan   


In [25]:

# prediction function
def prediction(lsh, X_train, y_train, X_query, k, metric):
    prediction_list = []

    for x in X_query:
        candidates =lsh.query(x)
        if len(candidates) == 0:
            prediction_list.append(y_train.mode()[0])
            continue
        X_cand = X_train[candidates]
        sims = sim_metric(x, X_cand, metric)
        sorted_idx = np.argsort(-sims)
        k_eff = min(k, len(candidates))   # handle case where lower k candidates exist
        top_k = sorted_idx[:k_eff]
        neighbor_lables = y_train.iloc[candidates].iloc[top_k]
        unique, counts = np.unique(neighbor_lables, return_counts=True)
        pred = unique[np.argmax(counts)]
        prediction_list.append(pred)

    return np.array(prediction_list)

We implemented the Achlioptas random projection scheme.


$$
r_{ij} =
\begin{cases}
+\sqrt{3} & \text{with probability } \frac{1}{6}, \\
0         & \text{with probability } \frac{2}{3}, \\
-\sqrt{3} & \text{with probability } \frac{1}{6}.
\end{cases}
$$
 which produces computationally efficient projection matrix.           

For each of the n hash tables, we create a new random projection matrix R.
Then for every track vector x, we compute its hash by projecting it and taking the sign: $$
h(x) = \text{sign}(R x)
$$ which decides to which bucket the track goes to.

For a query track, we first compute its hash in every hash table.
Then we collect all tracks that end up in the same buckets as the query — these are the candidate neighbors.

Next, we measure how similar the query is to each candidate using either cosine similarity or the negative of the Euclidean distance.
We pick the top‑k most similar tracks.

The genre is predicted by taking the majority vote among these k neighbors.
If there are fewer than k candidates, we just use all the ones we have.

*Subtask 2 :  Hyperparameter optimization*

In [26]:
# hyperameter serach

l_val = [8,16,32]
n_val = [5,10,20]
k_val = [3,5,10]

metrics = ["cosine", "euclidean"]
results = []

for l in l_val:
    for n in n_val:
        for k in k_val:
            for metric in metrics:

                print(f"Testing l={l}, n={n}, k={k}, metric={metric}")

                lsh = LSH(l, n)
                lsh.fit(X_train)
                y_pred = prediction(lsh, X_train, y_train, X_val, k, metric)
                acc = np.mean(y_pred == y_val.values)

                results.append((l,n,k,metric,acc))
                print("Accuracy:",acc)




Testing l=8, n=5, k=3, metric=cosine
Accuracy: 0.6524100475220638
Testing l=8, n=5, k=3, metric=euclidean
Accuracy: 0.6605566870332654
Testing l=8, n=5, k=5, metric=cosine
Accuracy: 0.6666666666666666
Testing l=8, n=5, k=5, metric=euclidean
Accuracy: 0.6673455532926001
Testing l=8, n=5, k=10, metric=cosine
Accuracy: 0.6802443991853361
Testing l=8, n=5, k=10, metric=euclidean
Accuracy: 0.6524100475220638
Testing l=8, n=10, k=3, metric=cosine
Accuracy: 0.6693822131704006
Testing l=8, n=10, k=3, metric=euclidean
Accuracy: 0.658520027155465
Testing l=8, n=10, k=5, metric=cosine
Accuracy: 0.681602172437203
Testing l=8, n=10, k=5, metric=euclidean
Accuracy: 0.6809232858112695
Testing l=8, n=10, k=10, metric=cosine
Accuracy: 0.6972165648336728
Testing l=8, n=10, k=10, metric=euclidean
Accuracy: 0.6707399864222675
Testing l=8, n=20, k=3, metric=cosine
Accuracy: 0.6897488119484046
Testing l=8, n=20, k=3, metric=euclidean
Accuracy: 0.6768499660556687
Testing l=8, n=20, k=5, metric=cosine
Accurac

In [27]:
# best parameters
best_parameters = max(results, key=lambda x:x[4])
best_l, best_n, best_k, best_metric, best_acc = best_parameters
print("\nBest Parameters:")
print(best_parameters)


Best Parameters:
(8, 20, 10, 'cosine', 0.7080787508486083)


In [28]:

# retrain on best parameters
X_train_full = np.vstack([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])
lsh = LSH(best_l, best_n)
lsh.fit(X_train_full)

In [29]:
# Final test evaluation
y_test_pred = prediction(lsh,X_train_full, y_train_full, X_test, best_k, best_metric)
test_acc = np.mean(y_test_pred == y_test.values)
print("\nFinal Test Accuracy:", test_acc)


Final Test Accuracy: 0.6950819672131148


Training and Evaluation Procedure

To evaluate the LSH‑based classifier, we tried every parameter setting by rebuilding the LSH structure on the training data and then using it to search for neighbors of each validation track. For each query, we calculated similarity scores using cosine similarity or negative Euclidean distance, selected the top‑k closest tracks, and predicted the genre through majority voting. After making predictions for all validation tracks, we measured the accuracy. We repeated this entire routine for every parameter combination to find the best setup.

Hyperparameters Tested

We tested the following values:

Hash length (l)     : 8, 16, 32

Number of hash tables (n): 5, 10, 20

Number of neighbors (k): 3, 5, 10

Similarity metric (m): cosine, euclidean

For each one, we trained the LSH model and evaluated its accuracy on the validation set.

Best Parameter Choice

After testing all combinations, the best performance on the validation set was achieved with:

l = 8

n = 20

k = 10

Similarity metric = Euclidean

Validation accuracy ≈ 0.7141

why we settled on a specific choice

l = 8:  
A shorter hash length caused more collisions, which produced more candidate neighbors and increased the chance of finding similar tracks.

n = 20:  
Using more hash tables made it more likely that similar tracks would land in the same bucket in at least one table. With fewer tables, many queries didn’t get enough candidates.

k = 10:  
Taking more neighbors made the voting process more reliable and reduced random noise.

Euclidean similarity:  
Euclidean distance (converted to similarity) worked better than cosine similarity on the standardized features, leading to higher accuracy.

After selecting the best parameters, we retrained the model on the combined training + validation data and evaluated it once on the test set.

Final Test Accuracy:
0.6957

This means the chosen hyperparameters also work well on new data, not just the data used during tuning.

*Subtask 3: Implementation details & runtime*

The Achlioptas projection is a quicker and more efficient substitute for a full Gaussian random matrix, while still keeping the distances between points mostly unchanged.
It is more practical for large dataset and easier to implement.

The Achlioptas projection is computationally cheaper and produces a sparse matrix, which reduces memory and computation time, yet it still preserves pairwise distances almost as well as a full Gaussian projection, making it a practical choice for large‑scale LSH.


*Subtask 4: Survey*

We each spent an average of three days on this assignment. We worked individually, and after completing our works, we exchanged our source code, discussed the strengths and weaknesses,  and finally agreed on a final version.